# Runbook 5.4.d: PKI and digital certificates  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Mitigate Risk · **Maps to:** 5.4.B, 5.4.C · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** A browser has just received a TLS certificate chain from a server: a leaf (end-entity) certificate
and one or more issuing CA certificates, and it needs to decide whether to trust it. Real browsers
call into a full X.509 library to do this; you are going to implement the *validation logic itself*, the same logic that library runs under the hood, using the small-number RSA sign/verify math from
5.4.b/5.4.c.

Each certificate is a plain `dict`:
```python
{
  "subject": "www.meridianbank.example",     # who this certificate identifies
  "issuer": "Meridian Intermediate CA",       # who issued (signed) it
  "public_key": [17, 2501],                   # [e, n] -- this cert's own public key
  "signature": 1271,                          # issuer's signature over this cert's fields
  "not_before": 1000,                         # validity window, integer "time units"
  "not_after": 9000,
  "revoked": False,
}
```

A **chain** is a list of these dicts ordered leaf-first: `[leaf, intermediate..., root]`. The last
entry is claimed to be a root CA certificate (self-signed: its own `issuer` equals its own
`subject`).

**Lesson objective.** Explain what a digital certificate contains and why, describe how a client walks a certificate chain up to a trusted root to decide whether to trust a public key, and identify why and how certificates get revoked before their stated expiration.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
validate_chain([leaf, intermediate, root], {"Meridian Root CA": [17, 2773]}, 5000)
  -> {"valid": True, "reason": "valid"}

validate_chain([expired_leaf, intermediate, root], {"Meridian Root CA": [17, 2773]}, 9500)
  -> {"valid": False, "reason": "expired"}
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only. No `cryptography`, no `ssl`, no `hashlib` -- this
- [ ] `trusted_roots` is the only source of truth for which root keys are trusted -- a self-signed
- [ ] Must pass all cases in the tests (visible + hidden): a fully valid chain, an expired leaf, a
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry points
```python
def compute_digest(cert: dict) -> int
def rsa_sign(digest: int, d: int, n: int) -> int
def rsa_verify(digest: int, signature: int, e: int, n: int) -> bool
def validate_chain(chain: list, trusted_roots: dict, current_time: int) -> dict
def solve(data: dict) -> dict
```

`solve()` is the blackbox entry point; the other functions are the underlying primitives it is
built on and are tested directly as well.

#### Input
- `compute_digest(cert)`: `cert` is a `dict` with keys `subject` (`str`), `issuer` (`str`),
  `public_key` (`[e, n]` list of two `int`), `not_before` (`int`), `not_after` (`int`). (`signature`
  and `revoked` may also be present but are not part of the digest.)
- `rsa_sign(digest, d, n)` / `rsa_verify(digest, signature, e, n)`: all `int`.
- `validate_chain(chain, trusted_roots, current_time)`:
  - `chain`: `list[dict]`, leaf-first, each dict shaped like `cert` above plus `signature` (`int`,
    the issuer's signature over `compute_digest(cert)`) and `revoked` (`bool`).
  - `trusted_roots`: `dict[str, list[int]]` mapping a trusted root CA's `subject` name to its known
    `[e, n]` public key (the local trust store).
  - `current_time`: `int`.
- `solve(data)`: `data` is a `dict` with `"op"` equal to `"validate"` or `"digest"`.
  - `"validate"`: `{"op": "validate", "chain": [...], "trusted_roots": {...}, "current_time": int}`
  - `"digest"`: `{"op": "digest", "cert": {...}}`

#### Output (`result`)
- `compute_digest(...)` returns an `int`.
- `rsa_sign(...)` returns an `int`.
- `rsa_verify(...)` returns `bool`.
- `validate_chain(...)` returns `{"valid": bool, "reason": str}`, where `reason` is one of
  `"valid"`, `"expired"`, `"revoked"`, `"broken_chain"`, `"untrusted_root"`, `"bad_signature"`.
- `solve(...)` returns `validate_chain(...)`'s dict for op `"validate"`, or `{"digest": int}` for op
  `"digest"`.

#### Guarantees / rules
- Pure functions, deterministic, no I/O, no network. Standard library only -- no `cryptography`,
  `ssl`, or `hashlib`; this is a from-scratch textbook-RSA exercise, not a real X.509 parser.
- `compute_digest` canonicalizes as
  `f"{subject}|{issuer}|{e}:{n}|{not_before}|{not_after}"` and reduces with the rolling hash
  `h = (h * 131 + ord(ch)) % (2**61 - 1)` over every character.
- `validate_chain` walks the chain leaf-first (index 0) to root (last index) and, for each
  certificate, checks in this order: validity window, revocation, issuer/subject chain linkage
  (or trusted-root membership for the last certificate), then signature -- returning the first
  failure encountered.
- The public key used to verify a non-root certificate's signature is always the *next*
  certificate's `public_key`, never the certificate's own key. The public key used to verify the
  root's self-signature is always looked up from `trusted_roots`, never trusted from the chain data
  itself.
- A self-signed certificate whose subject is absent from `trusted_roots` is `"untrusted_root"`, even
  if its own signature verifies correctly.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import Any

_MODULUS = (2 ** 61) - 1


def compute_digest(cert: dict) -> int:
    e, n = cert["public_key"]
    canonical = f'{cert["subject"]}|{cert["issuer"]}|{e}:{n}|{cert["not_before"]}|{cert["not_after"]}'
    h = 0
    for ch in canonical:
        h = (h * 131 + ord(ch)) % _MODULUS
    return h


def rsa_sign(digest: int, d: int, n: int) -> int:
    return pow(digest % n, d, n)


def rsa_verify(digest: int, signature: int, e: int, n: int) -> bool:
    return pow(signature, e, n) == digest % n


def validate_chain(chain: list, trusted_roots: dict, current_time: int) -> dict:
    for i, cert in enumerate(chain):
        if current_time < cert["not_before"] or current_time > cert["not_after"]:
            return {"valid": False, "reason": "expired"}

        if cert.get("revoked", False):
            return {"valid": False, "reason": "revoked"}

        if i + 1 < len(chain):
            issuer_cert = chain[i + 1]
            if cert["issuer"] != issuer_cert["subject"]:
                return {"valid": False, "reason": "broken_chain"}
            issuer_pubkey = issuer_cert["public_key"]
        else:
            root_key = trusted_roots.get(cert["subject"])
            if root_key is None or list(root_key) != list(cert["public_key"]):
                return {"valid": False, "reason": "untrusted_root"}
            issuer_pubkey = root_key

        digest = compute_digest(cert)
        e, n = issuer_pubkey
        if not rsa_verify(digest, cert["signature"], e, n):
            return {"valid": False, "reason": "bad_signature"}

    return {"valid": True, "reason": "valid"}


def solve(data: Any) -> Any:
    op = data["op"]
    if op == "validate":
        return validate_chain(data["chain"], data["trusted_roots"], data["current_time"])
    elif op == "digest":
        return {"digest": compute_digest(data["cert"])}
    else:
        raise ValueError(f"unknown op: {op!r}")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

import copy


_MODULUS = (2 ** 61) - 1


def _oracle_digest(cert: dict) -> int:
    e, n = cert["public_key"]
    canonical = f'{cert["subject"]}|{cert["issuer"]}|{e}:{n}|{cert["not_before"]}|{cert["not_after"]}'
    h = 0
    for ch in canonical:
        h = (h * 131 + ord(ch)) % _MODULUS
    return h


ROOT_E, ROOT_N, ROOT_D = 17, 2773, 157
INT_E, INT_N, INT_D = 17, 2537, 1433
LEAF_E, LEAF_N = 17, 2501

ROOT_CERT = {
    "subject": "Meridian Root CA",
    "issuer": "Meridian Root CA",
    "public_key": [ROOT_E, ROOT_N],
    "not_before": 1000,
    "not_after": 9000,
    "revoked": False,
    "signature": 1287,
}

INTERMEDIATE_CERT = {
    "subject": "Meridian Intermediate CA",
    "issuer": "Meridian Root CA",
    "public_key": [INT_E, INT_N],
    "not_before": 1000,
    "not_after": 9000,
    "revoked": False,
    "signature": 884,
}

LEAF_CERT = {
    "subject": "www.meridianbank.example",
    "issuer": "Meridian Intermediate CA",
    "public_key": [LEAF_E, LEAF_N],
    "not_before": 1000,
    "not_after": 9000,
    "revoked": False,
    "signature": 1271,
}

TRUSTED_ROOTS = {"Meridian Root CA": [ROOT_E, ROOT_N]}


def _chain():
    return [copy.deepcopy(LEAF_CERT), copy.deepcopy(INTERMEDIATE_CERT), copy.deepcopy(ROOT_CERT)]


def test_valid_chain_is_trusted():
    result = impl.validate_chain(_chain(), TRUSTED_ROOTS, current_time=5000)
    assert result == {"valid": True, "reason": "valid"}


def test_digest_matches_oracle():
    assert impl.compute_digest(LEAF_CERT) == _oracle_digest(LEAF_CERT)
    assert impl.compute_digest(INTERMEDIATE_CERT) == _oracle_digest(INTERMEDIATE_CERT)
    assert impl.compute_digest(ROOT_CERT) == _oracle_digest(ROOT_CERT)


def test_rsa_sign_and_verify_round_trip():
    digest = _oracle_digest(LEAF_CERT)
    sig = impl.rsa_sign(digest, INT_D, INT_N)
    assert sig == LEAF_CERT["signature"]
    assert impl.rsa_verify(digest, sig, INT_E, INT_N) is True


def test_expired_leaf_is_rejected():
    chain = _chain()
    result = impl.validate_chain(chain, TRUSTED_ROOTS, current_time=9500)
    assert result == {"valid": False, "reason": "expired"}


def test_revoked_intermediate_is_rejected():
    chain = _chain()
    chain[1]["revoked"] = True
    result = impl.validate_chain(chain, TRUSTED_ROOTS, current_time=5000)
    assert result == {"valid": False, "reason": "revoked"}


def test_solve_validate_op():
    result = impl.solve({
        "op": "validate",
        "chain": _chain(),
        "trusted_roots": TRUSTED_ROOTS,
        "current_time": 5000,
    })
    assert result == {"valid": True, "reason": "valid"}


def test_broken_issuer_subject_link_is_rejected():
    chain = _chain()
    chain[0]["issuer"] = "Some Other CA"
    result = impl.validate_chain(chain, TRUSTED_ROOTS, current_time=5000)
    assert result == {"valid": False, "reason": "broken_chain"}


def test_untrusted_self_signed_root_is_rejected():
    chain = _chain()
    result = impl.validate_chain(chain, trusted_roots={}, current_time=5000)
    assert result == {"valid": False, "reason": "untrusted_root"}


def test_root_with_mismatched_trusted_key_is_rejected():
    chain = _chain()
    tampered_roots = {"Meridian Root CA": [ROOT_E, ROOT_N + 2]}
    result = impl.validate_chain(chain, tampered_roots, current_time=5000)
    assert result == {"valid": False, "reason": "untrusted_root"}


def test_tampered_leaf_signature_is_rejected():
    chain = _chain()
    chain[0]["signature"] = chain[0]["signature"] + 1
    result = impl.validate_chain(chain, TRUSTED_ROOTS, current_time=5000)
    assert result == {"valid": False, "reason": "bad_signature"}


def test_tampered_leaf_subject_is_rejected_as_bad_signature():
    chain = _chain()
    chain[0]["subject"] = "www.evil.example"
    result = impl.validate_chain(chain, TRUSTED_ROOTS, current_time=5000)
    assert result == {"valid": False, "reason": "bad_signature"}


def test_expired_check_takes_priority_over_broken_chain():
    chain = _chain()
    chain[0]["not_after"] = 2000
    chain[0]["issuer"] = "Nonexistent CA"
    result = impl.validate_chain(chain, TRUSTED_ROOTS, current_time=5000)
    assert result == {"valid": False, "reason": "expired"}


def test_solve_digest_op():
    result = impl.solve({"op": "digest", "cert": LEAF_CERT})
    assert result == {"digest": _oracle_digest(LEAF_CERT)}


def test_current_time_at_exact_boundaries_is_valid():
    chain = _chain()
    result_start = impl.validate_chain(chain, TRUSTED_ROOTS, current_time=1000)
    result_end = impl.validate_chain(chain, TRUSTED_ROOTS, current_time=9000)
    assert result_start == {"valid": True, "reason": "valid"}
    assert result_end == {"valid": True, "reason": "valid"}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it